# Batch Processing: Batching, Backfills, Hash Partitioning, Compaction, Resumable Jobs

## What you'll practice

Batch jobs at scale run into the same handful of problems: request size limits (batch by count and bytes), missing history (backfills), uneven work (partitioning and skew),
too many small files (compaction), and long jobs that crash halfway (resumability). Each exercise is one of those problems in miniature.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/aws/test_11_03_batch_processing.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/11_python_for_aws/03_batch_processing.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
S3_ROOT = LAB / "datasets/s3"        # local stand-in for S3: <root>/<bucket>/<key>
for p in sorted(S3_ROOT.rglob("*"))[:12]:
    print(p.relative_to(S3_ROOT))

## Exercise 1 — 🟢 Easy · Batch by count and size

**Implement:** `plan_batches(items: list, max_items: int, max_bytes: int, size=len) -> list[list]`

Group items into consecutive batches respecting **both** limits (like Kinesis/Firehose `PutRecords`: ≤ 500 records, ≤ 5 MB). An item larger than `max_bytes` on its own → `ValueError`.

**Input:** `items: list, max_items: int, max_bytes: int, size=len`

**Expected output:** `list[list]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
plan_batches(['aa', 'bbb', 'c', 'dddd', 'e'], max_items=2, max_bytes=5)
# -> [['aa', 'bbb'], ['c', 'dddd'], ['e']]
```

**Check:** `pytest tests/aws/test_11_03_batch_processing.py -k test_plan_batches`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Plan a backfill

**Implement:** `plan_backfill(start: str, end: str, done: set[str], wave_size: int) -> list[list[str]]`

Return the missing dates (inclusive range, ISO strings, not in `done`) in chronological order, chunked into waves of at most `wave_size` (to run in parallel without overloading the cluster). `start > end` → `ValueError`.

**Input:** `start: str, end: str, done: set[str], wave_size: int`

**Expected output:** `list[list[str]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
plan_backfill('2026-01-30', '2026-02-03', {'2026-01-31'}, 2)
# -> [['2026-01-30', '2026-02-01'], ['2026-02-02', '2026-02-03']]
```

**Check:** `pytest tests/aws/test_11_03_batch_processing.py -k test_plan_backfill`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Deterministic hash partitioning and skew

**Implement:** `hash_partition(records: list[dict], key: str, n: int) -> list[list[dict]]`, `skew_ratio(partitions: list[list]) -> float`

Assign each record to partition `zlib.crc32(str(record[key]).encode()) % n` (deterministic across runs and machines — unlike Python's salted `hash()` for strings). `skew_ratio` = largest partition size /
mean size (rounded 2; empty input → 1.0). Try partitioning `orders.csv` by `customer_id` vs by `country` with n=8 and compare skew.

**Input:** `records: list[dict], key: str, n: int`

**Expected output:** `list[list[dict]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
skew_ratio([[1, 2, 3], [], [4]])
# -> 2.25
```

**Check:** `pytest tests/aws/test_11_03_batch_processing.py -k "test_hash_partition or test_skew_ratio"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Compact small files

**Implement:** `compact_partition(partition_dir, target_rows: int) -> dict`

Streaming jobs leave hundreds of tiny `part-*.csv` files per partition; readers (Athena/Spark) slow down. Rewrite all `part-*.csv` in `partition_dir` (same header required) into files of at most
`target_rows` rows named `part-c00000.csv`, `part-c00001.csv`, ... preserving row order (by source file name, then line). Build them in a staging directory, then swap: remove old part files,
move new ones in. Non-`part-` files (e.g. `_SUCCESS`) are kept. Return `{"files_before", "files_after", "rows"}`. Different headers → `ValueError` and **nothing changed**.

**Input:** `partition_dir, target_rows: int`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_compact_partition, test_compact_partition_header_mismatch` in `tests/aws/test_11_03_batch_processing.py`.

**Check:** `pytest tests/aws/test_11_03_batch_processing.py -k "test_compact_partition or test_compact_partition_header_mismatch"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · A resumable batch job · 🧩 Mini-project

**Implement:** `process_resumable(items: list, batch_size: int, process_batch, state_path, output_dir) -> dict`

Long jobs crash. Split `items` into batches; for each batch index not yet completed, call `process_batch(batch) -> list` and write the result atomically to `output_dir/batch-00000.json`
(etc.), then record the index as completed in a JSON state file and save it **after every batch**. If `process_batch` raises, let the exception propagate — completed batches stay recorded.
On the next call, completed batches are skipped. The state stores a fingerprint (SHA-256 of `json.dumps(items)` + batch_size); if a rerun uses different items/batch size → `ValueError`
(refuse to mix runs). Return `{"ran": [indexes], "skipped": [indexes], "total_batches": n}`. When everything is done, write `output_dir/_SUCCESS`.

**Input:** `items: list, batch_size: int, process_batch, state_path, output_dir`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_process_resumable` in `tests/aws/test_11_03_batch_processing.py`.

**Check:** `pytest tests/aws/test_11_03_batch_processing.py -k test_process_resumable`

**YOUR CODE ↓**